# Met Day App - Investigation

A document investigating the possiblity of creating an app that looks at the weather and determines if its a good day to go to the MET. The app will also let you search the art that is available to see and also create a  hunt type list of art to view

In [102]:
import requests
import pandas

Weather AP: Open Meteo (https://open-meteo.com/) 
- free/no signup 
- Trying to understand how to pull data and what data is available:

In [103]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 40.71,
    "longitude": -74.01
    #gets data for New York City, NY
}
#other parameters can be found here: https://open-meteo.com/en/docs

response = requests.get(url, params=params)
weather_data = response.json() #convert the response to a Python dictionary
print(weather_data)

{'latitude': 40.710335, 'longitude': -73.99308, 'generationtime_ms': 0.0015497207641601562, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 27.0}


In [104]:
#AI used here to help me understand how to use this API and get data I need. 
#Need to rquest the variables I want to see
#Data needs to requested by time (hourly, daily, current)
#For our purpose, I think we will need daily or current

params = {
    "latitude": 40.71,
    "longitude": -74.01,
    "current": ["temperature_2m", "weather_code", "wind_speed_10m", "is_day"],
    "daily": ["temperature_2m_max", "temperature_2m_min", "uv_index_max", "precipitation_sum"],
    "timezone": "auto"
}

response = requests.get(url, params=params)
weather_data = response.json() #convert the response to a Python dictionary
print(weather_data)

{'latitude': 40.710335, 'longitude': -73.99308, 'generationtime_ms': 0.34499168395996094, 'utc_offset_seconds': -14400, 'timezone': 'America/New_York', 'timezone_abbreviation': 'GMT-4', 'elevation': 27.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h', 'is_day': ''}, 'current': {'time': '2026-08-09T20:45', 'interval': 900, 'temperature_2m': 27.7, 'weather_code': 0, 'wind_speed_10m': 5.4, 'is_day': 0}, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'uv_index_max': '', 'precipitation_sum': 'mm'}, 'daily': {'time': ['2026-08-09', '2026-08-10', '2026-08-11', '2026-08-12', '2026-08-13', '2026-08-14', '2026-08-15'], 'temperature_2m_max': [34.1, 33.5, 32.3, 28.6, 29.6, 24.7, 25.7], 'temperature_2m_min': [22.7, 21.5, 21.5, 25.6, 22.2, 17.7, 18.4], 'uv_index_max': [7.4, 6.5, 6.75, 6.95, 6.8, 2.3, 7.1], 'precipitation_sum': [0.0, 4.7, 0.58, 0.4, 0.0, 2.9, 0.0]}}


In [105]:
#Also will need forecasted data

params = {
    "latitude": 40.71,
    "longitude": -74.01,
    "daily": ["temperature_2m_max", "precipitation_probability_max","temperature_2m_min", "precipitation_probability_min"],
    "forecast_days": 16,   # up to 16 max
    "timezone": "auto"
}

response = requests.get(url, params=params)
weather_data = response.json()
print(weather_data)

#Data is all in Celcius - may need to consider pulling in Farenheight
#can also consider looping through this data and charting it based on day


{'latitude': 40.710335, 'longitude': -73.99308, 'generationtime_ms': 0.2315044403076172, 'utc_offset_seconds': -14400, 'timezone': 'America/New_York', 'timezone_abbreviation': 'GMT-4', 'elevation': 27.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'precipitation_probability_max': '%', 'temperature_2m_min': '°C', 'precipitation_probability_min': '%'}, 'daily': {'time': ['2026-08-09', '2026-08-10', '2026-08-11', '2026-08-12', '2026-08-13', '2026-08-14', '2026-08-15', '2026-08-16', '2026-08-17', '2026-08-18', '2026-08-19', '2026-08-20', '2026-08-21', '2026-08-22', '2026-08-23', '2026-08-24'], 'temperature_2m_max': [34.1, 33.5, 32.3, 28.6, 29.6, 24.7, 25.7, 30.8, 26.4, 29.9, 27.8, 30.8, 27.9, 26.2, 29.2, 27.0], 'precipitation_probability_max': [2, 36, 7, 14, 38, 9, 5, 18, 28, 28, 15, 18, 27, 26, 27, 21], 'temperature_2m_min': [22.7, 21.5, 21.5, 25.6, 22.2, 17.7, 18.4, 22.3, 21.4, 22.1, 22.8, 23.3, 20.4, 21.7, 23.0, 18.7], 'precipitation_probability_min': [0, 0, 0, 3, 9, 

MET API: The Metropolitan Museum of Art Collection API (https://metmuseum.github.io/) 
- free/no signup or registration
- Trying to understand how to pull data and what data is available:

This data has several different tables that will need to call on one another. 
This is taken right from the metmuseum website.
1. Objects: A listing of all valid Object IDs available for access.
2. Object: A record for an object, containing all open access data about that object, including its image (if the image is available under Open Access)
3. Departments: A listing of all valid departments, with their department ID and the department display name
4. Search: A listing of all Object IDs for objects that contain the search query within the object’s data

In [106]:
met_search_url = "https://collectionapi.metmuseum.org/public/collection/v1/search"
met_search_params = {"q": "rain", "hasImages": "true"}
#other met search parameters can be found here: https://metmuseum.github.io/

#search data allows you to essentially filter through objects
met_response = requests.get(met_search_url, params=met_search_params)
met_data = met_response.json()

#see how many results can match a certain key word and have an image associated with it
print("total results:", met_data["total"])
#in the future may have to also call on the "is on display" parameter 
#not sure how updated this data is - looks to be last updated in 2020

#get the first few object ids to see what they look like
print("first few object ids:", met_data["objectIDs"][:5])


total results: 201
first few object ids: [57416, 310364, 817809, 551786, 729644]


In [107]:
#display some amount of information about the first object in the search results
#here we have to connect to the object table
first_object_id = met_data["objectIDs"][0]

object_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{first_object_id}"
object_response = requests.get(object_url)
object_data = object_response.json()

print("Title: ", object_data["title"])
print("Artist: ", object_data["artistDisplayName"])
print("Date: ", object_data["objectDate"])

#we can use the url to display the image similar to what we learned in class
print("Image: ", object_data["primaryImage"])

#ultimately we could create a list and search through that list to match the object ids
#then we could display the image and information about the object that matches creating a top 10 list
#could generate a list of random object ids for our "scavenger hunt" feature

Title:  Bowl decorated with cranes, rain, and chrysanthemums
Artist:  
Date:  early 14th century
Image:  https://images.metmuseum.org/CRDImages/as/original/DP-41796-001.jpg


Testing the option to return a recommendation on if someone should go to the MET
Based on some simple logic

In [109]:
today_high = data['daily']['temperature_2m_max'][0]
rain_chance = data['daily']['precipitation_probability_max'][0]
print(today_high, rain_chance)

def get_recommendation(high_temp, rain_chance):
    if rain_chance >= 50:
        return True, "rainy day, perfect excuse for the museum", "rain"
    elif high_temp >= 90:
        return True, "too hot to be outside for long", "sun"
    elif high_temp <= 35:
        return True, "too cold out, stay warm inside", "winter"
    else:
        return False, "actually pretty nice out today", "garden"

should_go, reason, search_term = get_recommendation(today_high, rain_chance)

print(should_go, "-", reason)
print("search term:", search_term)

#need to add some test cases here to make sure it works correctly - can add to test file

34.1 2
True - too cold out, stay warm inside
search term: winter


## The app should:
1. get weather for NYC on a specific day
2. decide if it's a good museum day based on temp/rain
3. search the Met for something themed to the weather??
4. second tab to search for art to view
5. third tab to generate random scavenger hunt for day at met

I've tested here:
- weather lookup - works and pulls 16 day forecast
- Met search + object detail lookup

To Do:
- put everything together
- maybe connect met opening hours? TBD
- maybe include date picker? TBD
- build the html/front end
- figure out how to change temperature to fareinheight
- deeper look into what parameters and variables are available to see what we can pull
